# **Feature Scaling/Engineering:**

**Scaling all features into same dimension/range:**
> importing data from previous file

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import tensorflow as tf
import keras
from sklearn.model_selection import train_test_split

In [3]:
symbol="BTC-USD"
df=pd.DataFrame(data=yf.download(tickers=symbol,period="1mo",interval="5m",auto_adjust=True)["Close"])
df["LR"]=np.log(df[symbol]/df[symbol].shift(1))
features=["Direction","SMA",'BB','rolling_min','rolling_max','momentum','Volatility']
sma_s=7;sma_l=30;sma_window=5
df[f"{features[0]}"]=np.where(df["LR"]>0,1,0)
df[f"{features[1]}"]=df[symbol].rolling(window=sma_s).mean()-df[symbol].rolling(window=sma_l).mean()
df[f"{features[2]}"]=(df[symbol]-df[symbol].rolling(window=sma_window).mean())/df[symbol].rolling(window=sma_window).std()
df[f"{features[3]}"]=df[symbol].rolling(window=sma_window).min()/(df[symbol]-1)
df[f"{features[4]}"]=df[symbol].rolling(window=sma_window).max()/(df[symbol]-1)
df[f"{features[5]}"]=df["LR"].rolling(window=sma_window).mean()
df[f"{features[6]}"]=df["LR"].rolling(window=sma_window).std()
df.dropna(inplace=True)
df

[*********************100%***********************]  1 of 1 completed


Ticker,BTC-USD,LR,Direction,SMA,BB,rolling_min,rolling_max,momentum,Volatility
Datetime,,,,,,,,,
2026-01-19 17:00:00+00:00,93137.859375,-0.001725,0,192.314360,-0.746831,1.000011,1.001737,-0.000107,0.001068
2026-01-19 17:05:00+00:00,93200.453125,0.000672,1,179.219940,0.067066,0.999339,1.001065,0.000102,0.001105
2026-01-19 17:10:00+00:00,93309.804688,0.001173,1,178.582254,1.111387,0.998168,1.000011,0.000334,0.001199
2026-01-19 17:15:00+00:00,93297.937500,-0.000127,0,183.224330,0.642384,0.998295,1.000138,0.000231,0.001215
2026-01-19 17:20:00+00:00,93265.500000,-0.000348,0,182.996615,0.321191,0.998642,1.000486,-0.000071,0.001108
...,...,...,...,...,...,...,...,...,...
2026-02-19 14:10:00+00:00,66061.375000,0.002586,1,-386.057515,1.778685,0.997278,1.000015,0.000530,0.001175
2026-02-19 14:15:00+00:00,65973.679688,-0.001328,0,-367.145982,0.427251,0.998604,1.001344,0.000218,0.001449
2026-02-19 14:20:00+00:00,65985.390625,0.000177,1,-336.828720,0.335341,0.998581,1.001167,0.000318,0.001420


In [4]:
lags=5;cols=[]
for i in features:
  for j in range(1,lags+1):
    col=f"{i}_lag_{j}"
    df[col]=df[i].shift(periods=j)
    cols.append(col)
df.dropna(inplace=True);df

Ticker,BTC-USD,LR,Direction,SMA,BB,rolling_min,rolling_max,momentum,Volatility,Direction_lag_1,...,momentum_lag_1,momentum_lag_2,momentum_lag_3,momentum_lag_4,momentum_lag_5,Volatility_lag_1,Volatility_lag_2,Volatility_lag_3,Volatility_lag_4,Volatility_lag_5
Datetime,,,,,,,,,,,,,,,,,,,,,
2026-01-19 17:25:00+00:00,93257.843750,-0.000082,0,176.378534,-0.198108,0.999395,1.000568,0.000257,0.000640,0.0,...,-0.000071,0.000231,0.000334,0.000102,-0.000107,0.001108,0.001215,0.001199,0.001105,0.001068
2026-01-19 17:30:00+00:00,93167.679688,-0.000967,0,149.130543,-1.648551,1.000011,1.001536,-0.000070,0.000779,0.0,...,0.000257,-0.000071,0.000231,0.000334,0.000102,0.000640,0.001108,0.001215,0.001199,0.001105
2026-01-19 17:35:00+00:00,93141.226562,-0.000284,0,142.521987,-1.252613,1.000011,1.001693,-0.000362,0.000356,0.0,...,-0.000070,0.000257,-0.000071,0.000231,0.000334,0.000779,0.000640,0.001108,0.001215,0.001199
2026-01-19 17:40:00+00:00,93100.703125,-0.000435,0,121.246726,-1.182663,1.000011,1.001781,-0.000423,0.000331,0.0,...,-0.000362,-0.000070,0.000257,-0.000071,0.000231,0.000356,0.000779,0.000640,0.001108,0.001215
2026-01-19 17:45:00+00:00,93163.687500,0.000676,1,92.034821,-0.044026,0.999335,1.001021,-0.000218,0.000598,0.0,...,-0.000423,-0.000362,-0.000070,0.000257,-0.000071,0.000331,0.000356,0.000779,0.000640,0.001108
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-19 14:10:00+00:00,66061.375000,0.002586,1,-386.057515,1.778685,0.997278,1.000015,0.000530,0.001175,0.0,...,-0.000343,-0.000719,-0.000908,-0.000855,-0.001069,0.000839,0.001092,0.000952,0.000999,0.000804
2026-02-19 14:15:00+00:00,65973.679688,-0.001328,0,-367.145982,0.427251,0.998604,1.001344,0.000218,0.001449,1.0,...,0.000530,-0.000343,-0.000719,-0.000908,-0.000855,0.001175,0.000839,0.001092,0.000952,0.000999
2026-02-19 14:20:00+00:00,65985.390625,0.000177,1,-336.828720,0.335341,0.998581,1.001167,0.000318,0.001420,0.0,...,0.000218,0.000530,-0.000343,-0.000719,-0.000908,0.001449,0.001175,0.000839,0.001092,0.000952


In [6]:
x_train,x_test,y_train,y_test=train_test_split(df.iloc[:,:],df.iloc[:,0],train_size=0.75,random_state=42)
train_test=[x_train,x_test,y_train,y_test]
for i in train_test:
  print(f"shape = {i.shape}")

shape = (6669, 44)
shape = (2224, 44)
shape = (6669,)
shape = (2224,)


**Filtering out mean & std of each features(with & without lags) from `x_train` & standardizing it using `(x_train-mu)/std`**

In [12]:
mu,std=x_train.mean(),x_train.std()
x_train_stand=(x_train-mu)/std
x_train_stand.describe().round(4)

Ticker,BTC-USD,LR,Direction,SMA,BB,rolling_min,rolling_max,momentum,Volatility,Direction_lag_1,...,momentum_lag_1,momentum_lag_2,momentum_lag_3,momentum_lag_4,momentum_lag_5,Volatility_lag_1,Volatility_lag_2,Volatility_lag_3,Volatility_lag_4,Volatility_lag_5
count,6669.0000,6669.0000,6669.0000,6669.0000,6669.0000,6669.0000,6669.0000,6669.0000,6669.0000,6669.0000,...,6669.0000,6669.0000,6669.0000,6669.0000,6669.0000,6669.0000,6669.0000,6669.0000,6669.0000,6669.0000
mean,0.0000,-0.0000,-0.0000,0.0000,-0.0000,0.0000,-0.0000,-0.0000,-0.0000,-0.0000,...,0.0000,0.0000,-0.0000,-0.0000,-0.0000,-0.0000,-0.0000,0.0000,-0.0000,0.0000
std,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,...,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
min,-1.7993,-9.5486,-0.9992,-5.4518,-1.7211,-14.6899,-0.6603,-9.9940,-1.1523,-0.9905,...,-8.1924,-9.8620,-9.8068,-9.9515,-9.9661,-1.1478,-1.1482,-1.1445,-1.1553,-1.1682
25%,-0.9150,-0.3820,-0.9992,-0.3791,-0.9061,-0.2364,-0.6587,-0.3715,-0.6404,-0.9905,...,-0.3644,-0.3707,-0.3678,-0.3692,-0.3816,-0.6384,-0.6391,-0.6388,-0.6410,-0.6491
50%,-0.1421,0.0244,-0.9992,0.0594,-0.0001,0.3367,-0.3325,0.0324,-0.2648,-0.9905,...,0.0361,0.0307,0.0263,0.0313,0.0302,-0.2648,-0.2693,-0.2675,-0.2696,-0.2675
75%,1.1654,0.3897,1.0007,0.4761,0.8984,0.6667,0.2088,0.4214,0.3129,1.0094,...,0.4277,0.4254,0.4192,0.4226,0.4212,0.3083,0.3093,0.3032,0.3098,0.3140
max,1.7154,11.1367,1.0007,5.0733,1.7587,0.6688,13.1430,8.7682,10.4950,1.0094,...,7.9328,8.6443,8.5927,8.7244,8.7350,10.4417,10.4047,10.3639,10.4517,10.6065
